# Lasso regression

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
warnings.simplefilter("ignore")

## One input: the slope reaches exactly 0

In [2]:
X, y = make_regression(n_samples=100, n_features=1, n_informative=1, noise=20, random_state=13)
for alpha in [0, 1, 5, 10, 20, 25, 30]:
    model = LinearRegression() if alpha == 0 else Lasso(alpha=alpha)
    model.fit(X, y)
    print(f"alpha {alpha:2d}: slope {model.coef_[0]:6.2f}, intercept {model.intercept_:.2f}")
xc, yc = X.ravel() - X.mean(), y - y.mean()
print("slope is 0 from alpha =", round((xc * yc).mean(), 2))

alpha  0: slope  27.83, intercept -2.29
alpha  1: slope  26.68, intercept -2.23
alpha  5: slope  22.07, intercept -1.96
alpha 10: slope  16.31, intercept -1.62
alpha 20: slope   4.80, intercept -0.95
alpha 25: slope   0.00, intercept -0.67
alpha 30: slope   0.00, intercept -0.67
slope is 0 from alpha = 24.17


## Degree-16 polynomial: which powers survive?

In [3]:
rng = np.random.default_rng(42)
x = 5 * rng.random((100, 1)) - 2
y2 = 0.7 * x[:, 0] ** 2 - 2 * x[:, 0] + 3 + rng.standard_normal(100)
for alpha in [0.01, 0.1, 1]:
    model = make_pipeline(PolynomialFeatures(degree=16, include_bias=False), StandardScaler(),
                          Lasso(alpha=alpha, max_iter=200000)).fit(x, y2)
    kept = [f"x^{p + 1}" for p in np.nonzero(model[-1].coef_)[0]]
    print(f"alpha {alpha}: {len(kept)} non-zero ->", kept)

alpha 0.01: 4 non-zero -> ['x^1', 'x^2', 'x^3', 'x^16']
alpha 0.1: 2 non-zero -> ['x^1', 'x^2']
alpha 1: 1 non-zero -> ['x^1']


## Diabetes: coefficients and feature selection

In [4]:
data = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=2)
rows = {}
for alpha in [0, 0.001, 0.01, 0.1, 0.5, 1, 2, 5]:
    reg = LinearRegression() if alpha == 0 else Lasso(alpha=alpha, max_iter=100000)
    reg.fit(X_train, y_train)
    rows[alpha] = list(reg.coef_.round(1)) + [int((reg.coef_ == 0).sum()), round(reg.score(X_test, y_test), 3)]
pd.DataFrame(rows, index=data.feature_names + ["zeros", "test R2"]).T

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,zeros,test R2
0.000,-9.2,-205.5,516.7,340.6,-895.6,561.2,153.9,126.7,861.1,52.4,0.0,0.440
0.001,-8.3,-204.2,517.7,339.7,-826.7,508.6,120.9,113.9,836.3,52.0,0.0,0.440
0.010,-1.4,-192.9,526.4,332.6,-430.2,191.3,-44.0,69.0,688.4,47.9,0.0,0.441
0.100,0.0,-114.0,526.7,292.6,-82.7,-0.0,-152.7,0.0,551.1,7.2,3.0,0.433
0.500,0.0,-0.0,473.2,155.4,-0.0,-0.0,-22.2,0.0,448.9,0.0,6.0,0.389
1.000,0.0,0.0,363.9,27.3,0.0,0.0,-0.0,0.0,336.1,0.0,7.0,0.326
2.000,0.0,0.0,62.1,0.0,0.0,0.0,0.0,0.0,43.5,0.0,8.0,0.057
5.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,-0.013


## Bias and variance against alpha (fresh noise, known true curve)

In [5]:
# Toy data, because bias and variance need the true curve. 20 fixed training x, fresh noise for each of 200 draws.
f = lambda x: 0.7 * x ** 2 - 2 * x + 3
SD = 2.0
x_train = np.linspace(-2, 3, 20)[:, None]
x_test = (x_train[1:] + x_train[:-1]) / 2          # test x between the training x

def bias_variance(alpha, draws=200):
    r = np.random.default_rng(1)
    preds = []
    for _ in range(draws):
        yy = f(x_train[:, 0]) + SD * r.standard_normal(len(x_train))
        model = make_pipeline(PolynomialFeatures(degree=16, include_bias=False), StandardScaler(),
                              Lasso(alpha=alpha, max_iter=100000))
        preds.append(model.fit(x_train, yy).predict(x_test))
    preds = np.array(preds)
    bias2 = ((preds.mean(0) - f(x_test[:, 0])) ** 2).mean()
    var = preds.var(0).mean()
    return bias2 + var + SD ** 2, bias2, var

for alpha in [0.001, 0.01, 0.1, 0.3, 1, 3]:
    loss, bias2, var = bias_variance(alpha)
    print(f"alpha {alpha:>6}: error {loss:5.2f}  bias2 {bias2:5.3f}  variance {var:5.2f}")

alpha  0.001: error  5.45  bias2 0.009  variance  1.44


alpha   0.01: error  4.96  bias2 0.009  variance  0.95


alpha    0.1: error  4.66  bias2 0.039  variance  0.62


alpha    0.3: error  4.88  bias2 0.339  variance  0.54


alpha      1: error  6.90  bias2 2.519  variance  0.38


alpha      3: error  9.40  bias2 5.203  variance  0.19
